# Data Loading (ETL)

## Assignment

### Dataset

| Order_ID | Customer_ID | Sales_Amount | Order_Date |
|---|---|---|---|
| O101 | C001 | 4500 | 12-01-2024 |
| O102 | C002 | NULL | 15-01-2024 |
| O103 | C003 | 3200 | 2024/01/18 |
| O101 | C001 | 4500 | 12-01-2024 |
| O104 | C004 | Three Thousand | 20-01-2024 |
| O105 | C005 | 5100 | 25-01-2024 |

## Q1. Data Understanding

**Question:** Identify all data quality issues present in the dataset that can cause problems during data loading.

### Solution

The dataset contains the following data quality issues:

1. **Duplicate Order_ID:** `O101` appears twice. Since Order_ID is assumed to be the Primary Key, this violates the uniqueness rule.
2. **Missing Sales Amount:** Order `O102` has a NULL value in the `Sales_Amount` column.
3. **Invalid Data Type:** Order `O104` contains `"Three Thousand"` in `Sales_Amount`, while the column is expected to contain numeric values.
4. **Inconsistent Date Format:** Most dates use `DD-MM-YYYY`, but Order `O103` uses `YYYY/MM/DD`.
5. **Duplicate Record:** The complete record for Order `O101` is repeated.

These issues should be corrected before loading the data into the target database.

## Q2. Primary Key Validation

**Question:** Assume `Order_ID` is the Primary Key.  
a) Is the dataset violating the Primary Key rule?  
b) Which record(s) cause this violation?

### Solution

**a)** Yes, the dataset violates the Primary Key rule.

A Primary Key must contain unique and non-null values. `Order_ID = O101` occurs twice, so uniqueness is violated.

**b)** The two records with `Order_ID = O101` cause the violation:

| Order_ID | Customer_ID | Sales_Amount | Order_Date |
|---|---|---:|---|
| O101 | C001 | 4500 | 12-01-2024 |
| O101 | C001 | 4500 | 12-01-2024 |

One duplicate record should be removed before loading.

## Q3. Missing Value Analysis

**Question:** Which column(s) contain missing values? List the affected records. Explain why loading these records without handling missing values is risky.

### Solution

The `Sales_Amount` column contains a missing value.

The affected record is:

| Order_ID | Customer_ID | Sales_Amount | Order_Date |
|---|---|---|---|
| O102 | C002 | NULL | 15-01-2024 |

Loading this record without handling the missing value is risky because `Sales_Amount` is important for calculating total sales, average sales, revenue reports, and BI dashboards.

If NULL values are not handled properly, calculations may become incomplete or misleading. The missing value should therefore be investigated and either recovered from the source, imputed according to an approved business rule, or flagged before loading.

## Q4. Data Type Validation

**Question:** Identify records where `Sales_Amount` violates expected data type rules.  
a) Which record(s) will fail numerical validation?  
b) What would happen if this dataset is loaded into a SQL table with `Sales_Amount` as DECIMAL?

### Solution

**a)** Order `O104` will fail numerical validation because its `Sales_Amount` is stored as text:

| Order_ID | Customer_ID | Sales_Amount | Order_Date |
|---|---|---|---|
| O104 | C004 | Three Thousand | 20-01-2024 |

The correct numeric representation should be `3000`.

**b)** If `Sales_Amount` is defined as a DECIMAL column in SQL, the text value `"Three Thousand"` cannot normally be converted to DECIMAL. The database may reject the row or the load operation may fail with a data conversion/type error, depending on the database and loading configuration.

Therefore, the value should be converted to `3000` before loading.

## Q5. Date Format Consistency

**Question:** The `Order_Date` column has multiple formats.  
a) List all date formats present in the dataset.  
b) Why is this a problem during data loading?

### Solution

**a)** Two date formats are present:

1. `DD-MM-YYYY` — Example: `12-01-2024`
2. `YYYY/MM/DD` — Example: `2024/01/18`

**b)** Inconsistent date formats can cause parsing errors or incorrect date interpretation during loading. A target database expects dates to be interpreted consistently.

Before loading, all dates should be converted to one standard format. A suitable standard is ISO format:

`YYYY-MM-DD`

For example:

`12-01-2024` → `2024-01-12`  
`2024/01/18` → `2024-01-18`

## Q6. Load Readiness Decision

**Question:** Based on the dataset condition:  
a) Should this dataset be loaded directly into the database? (Yes/No)  
b) Justify your answer with at least three reasons.

### Solution

**a) No**, this dataset should not be loaded directly into the database.

**b)** The dataset should first be cleaned and validated because:

1. `Order_ID O101` is duplicated and violates the Primary Key rule.
2. `Sales_Amount` is missing for Order `O102`.
3. `Sales_Amount` contains the text value `"Three Thousand"` for Order `O104`, which violates the expected numeric data type.
4. `Order_Date` contains inconsistent date formats.

Loading the dataset without correcting these issues may cause database errors and inaccurate reports.

## Q7. Pre-Load Validation Checklist

**Question:** List the exact pre-load validation checks you would perform on this dataset before loading.

### Solution

The following pre-load validation checks should be performed:

1. **Primary Key Check:** Verify that every `Order_ID` is unique and not NULL.
2. **Duplicate Check:** Identify and remove duplicate records.
3. **Missing Value Check:** Check mandatory columns such as `Order_ID`, `Customer_ID`, `Sales_Amount`, and `Order_Date` for NULL values.
4. **Data Type Check:** Ensure `Sales_Amount` contains only valid numeric values.
5. **Date Format Check:** Convert all `Order_Date` values into one standard date format.
6. **Customer ID Validation:** Ensure `Customer_ID` follows the required format and, where a customer master exists, references a valid customer.
7. **Range/Business Rule Check:** Ensure `Sales_Amount` is valid according to business rules, such as being non-negative.
8. **Record Count Check:** Compare source and cleaned record counts and document records removed or rejected.
9. **Final Schema Check:** Confirm that column names and data types match the target database schema.

## Q8. Cleaning Strategy

**Question:** Describe the step-by-step cleaning actions required to make this dataset load-ready.

### Solution

The dataset can be made load-ready using the following steps:

1. **Remove the duplicate record:** Keep only one occurrence of `Order_ID O101`.
2. **Handle the missing Sales_Amount:** Investigate the source for Order `O102`. If the correct amount can be recovered, replace the NULL. Otherwise, handle or flag it according to the approved business rule.
3. **Correct the invalid Sales_Amount:** Convert `"Three Thousand"` for Order `O104` into the numeric value `3000`.
4. **Standardize dates:** Convert every `Order_Date` into `YYYY-MM-DD`.
5. **Convert data types:** Ensure `Sales_Amount` is numeric/DECIMAL and `Order_Date` is a proper DATE type.
6. **Validate Primary Key:** Confirm that all remaining `Order_ID` values are unique and non-null.
7. **Run final quality checks:** Check for remaining NULLs, invalid values, duplicates, and schema mismatches.
8. **Load the validated data:** After all checks pass, load the cleaned records into the target database.

## Q9. Loading Strategy Selection

**Question:** Assume this dataset represents daily sales data.  
a) Should a Full Load or Incremental Load be used?  
b) Justify your choice.

### Solution

**a)** An **Incremental Load** should generally be used.

**b)** Since the dataset represents daily sales data, new transactions are expected to be added regularly. An incremental load processes only new or changed records instead of reloading the complete historical sales dataset every day.

This approach is more efficient because it:

- Reduces processing time.
- Reduces database and network load.
- Avoids unnecessarily reprocessing historical records.
- Scales better as the sales dataset becomes larger.

A Full Load may be appropriate for the initial load or for a very small dataset, but regular daily sales processing is better suited to Incremental Loading.

## Q10. BI Impact Scenario

**Question:** Assume this dataset was loaded without cleaning and connected to a BI dashboard.  
a) What incorrect results might appear in Total Sales KPI?  
b) Which records specifically would cause misleading insights?  
c) Why would BI tools not detect these issues automatically?

### Solution

**a)** The Total Sales KPI may be incorrect because:

- The duplicate `O101` transaction would count the same ₹4,500 sale twice and overstate sales by ₹4,500.
- The NULL `Sales_Amount` for `O102` may be ignored by aggregation, making the sales total incomplete.
- `"Three Thousand"` for `O104` may be treated as text, rejected, converted incorrectly, or excluded from numeric aggregation depending on the system.

If a BI tool sums only the valid numeric values as loaded, the visible numeric total could be misleading because of both duplication and excluded/invalid values.

**b)** The records causing misleading insights are:

- `O101` — duplicated transaction.
- `O102` — missing Sales_Amount.
- `O104` — non-numeric Sales_Amount.
- `O103` — inconsistent date format, which could affect date-based analysis if parsed incorrectly.

**c)** BI tools mainly visualize and aggregate the data they receive. They do not automatically know the organization's business rules or whether a repeated record is a genuine second transaction, whether a NULL value is an error, or which date format was intended.

Therefore, data quality and validation should be handled in the ETL pipeline before the data reaches the BI dashboard.

# Conclusion

The dataset is **not load-ready in its original form** because it contains a duplicate Primary Key, a missing sales value, an invalid numeric value, and inconsistent date formatting.

Before loading, the data should be cleaned, standardized, validated against the target schema and business rules, and checked again for duplicates and missing values. Since the dataset represents daily sales, an **Incremental Load** is the preferred ongoing loading strategy after the initial clean load.